# AutoEncoders

An autoencoder is a type of neural network designed to learn efficient representations of data (typically for dimensionality reduction or feature learning) by training the network to ignore "noise." It works by compressing the input into a latent-space representation and then reconstructing the output from this representation.


### The Architecture: Encoding and Decoding

An autoencoder consists of two main components connected by a "bottleneck."

1. The Encoder

The encoder's job is to compress the input data into a lower-dimensional code. It passes the input through a series of layers that gradually decrease in size.

- The Process: It maps the input x to a latent representation z (also called a "bottleneck" or "hidden state").

- The Math: This can be represented as z=f(x).

2. The Bottleneck (Latent Space)

This is the most important part of the network. By forcing the data through a layer with fewer dimensions than the input, the network is restricted. It cannot simply "copy" the input to the output; it must prioritize which aspects of the data are most important to keep.

3. The Decoder

The decoder's job is to take the compressed code z and reconstruct the original data as accurately as possible.

- The Process: It maps the latent representation z back to a reconstruction x′.

- The Math: This is represented as x′=g(z).


### AutoEncoders are useful in different tasks like:

- Dimensionality Reduction:
    - Unlike PCA that can be used for reducing complex data into fewer essential variables but only works for linear model, Autoencoders work on complex, non-linear data.

- Denoising:
    - By training a network on "noisy" data and telling it the target is the "clean" version, the autoencoder learns to strip away the noise and keep only the signal.

- Anomaly Detection:
    - If you train autoencoder on "normal" data, it becomes good at reconstructing that data, but it encounters abnormal data, the reconstruction error will be high, signalling an anomaly.

- Generative Modeling:
    - VAE (Variational Autoencoders) allow us to sample from the latent space to generate entirely new data, such as realistic faces or synthetic music.

- Feature Extraction:
    - Once trained, the encoder part of Autoencoders can be used as a "feature extractor" for other machine learing tasks, like classification.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch.utils.data import TensorDataset, DataLoader

def draw_circle(radius, center_x=0.5, center_y=0.5, size=28):
    # draw a circle using coordinates for the center, and the radius
    circle = plt.Circle((center_x, center_y), radius, color='k', fill=False)
    fig, ax = plt.subplots(figsize=(1, 1))
    ax.add_patch(circle)
    ax.axis('off')
    buf = fig.canvas.print_to_buffer()
    plt.close()
    # converts matplotlib figure into PIL image, make it grayscale, and resize it
    return np.array(Image.frombuffer('RGBA', buf[1], buf[0]).convert('L').resize((int(size), int(size))))

def gen_circles(n, size=28):
    # generates random coordinates around (0.5, 0.5) as center points
    center_x = np.random.uniform(0.0, 0.03, size=n).reshape(-1, 1)+.5
    center_y = np.random.uniform(0.0, 0.03, size=n).reshape(-1, 1)+.5
    # generates random radius sizes between 0.03 and 0.47
    radius = np.random.uniform(0.03, 0.47, size=n).reshape(-1, 1)
    sizes = np.ones((n, 1))*size

    coords = np.concatenate([radius, center_x, center_y, sizes], axis=1)
    # generates circles using draw_circle function
    circles = np.apply_along_axis(func1d=lambda v: draw_circle(*v), axis=1, arr=coords)
    return circles, radius

np.random.seed(42)
# generates 1,000 circles
circles, radius = gen_circles(1000)

circles_ds = TensorDataset(torch.as_tensor(circles).unsqueeze(1).float()/255, torch.as_tensor(radius))
circles_dl = DataLoader(circles_ds, batch_size=32, shuffle=True, drop_last=True)

## Encoder model

In [2]:
import torch.nn as nn

def set_seed(self, seed=42):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.manual_seed(seed)
    np.random.seed(seed)

class Encoder(nn.Module):
    def __init__(self, input_shape, z_size, base_model):
        super().__init__()
        self.input_shape = input_shape
        self.z_size = z_size
        self.base_model = base_model
    
        output_size = self._get_output_size()
        self.lin_latent = nn.Linear(output_size, z_size)
    
    def _get_output_size(self):
        device = next(self.base_model.parameters()).device.type
        dummy = torch.zeros(1, *self.input_shape, device=device)
        size = self.base_model(dummy).size(1)
        return size

    def forward(self, x):
        base_out = self.base_model(x)
        out = self.lin_latent(base_out)
        return out

set_seed(13)

z_size = 1
input_shape = (1, 28, 28)

base_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(np.prod(input_shape), 2048),
    nn.LeakyReLU(),
    nn.Linear(2048, 2048),
    nn.LeakyReLU(),
)

encoder = Encoder(input_shape, z_size, base_model)

In [3]:
encoder._get_output_size()

2048

In [4]:
x, _ = circles_ds[7]
z = encoder(x)
z

tensor([[-0.1209]], grad_fn=<AddmmBackward0>)

### Decoder architecture

In [5]:
decoder = nn.Sequential(
    nn.Linear(z_size, 2048),
    nn.LeakyReLU(),
    nn.Linear(2048, 2048),
    nn.LeakyReLU(),
    nn.Linear(2048, np.prod(input_shape)),
    nn.Unflatten(1, input_shape)
)

In [6]:
x_tilde = decoder(z)
x_tilde, x_tilde.shape

(tensor([[[[ 1.9079e-01, -4.3900e-02, -4.9170e-02,  5.2142e-02, -8.0119e-02,
            -1.6324e-01,  3.8319e-02,  6.2965e-02, -3.7442e-02, -3.6085e-02,
             2.2930e-02, -1.2089e-01,  2.0558e-01,  1.3671e-01,  1.4607e-03,
             1.1066e-02, -1.3429e-01, -3.7842e-02,  6.1736e-02, -3.0217e-02,
            -7.4171e-02, -1.6376e-02, -6.4663e-02, -1.5638e-01, -9.6261e-02,
             5.3312e-02,  6.6354e-02, -2.6916e-02],
           [ 1.8874e-01,  9.7503e-02, -1.3948e-01, -1.2955e-01, -1.2210e-02,
             5.6815e-02, -7.5753e-02,  5.3483e-02,  6.4153e-02, -1.6740e-01,
            -5.0190e-02,  6.2855e-02,  9.7707e-02, -2.2777e-02, -1.1442e-01,
             1.6079e-01, -1.4634e-01,  2.0068e-01, -2.7668e-02,  6.3487e-02,
            -9.6757e-02,  3.0803e-02, -7.0600e-02,  1.1162e-01,  8.6569e-02,
            -1.5205e-02,  1.9754e-01,  1.1691e-01],
           [-7.3312e-02,  9.0082e-02, -4.4267e-02,  1.1471e-01,  6.9005e-02,
             1.5237e-03, -4.2527e-02,  1.4255e-01

## AutoEncoder

In [7]:
class AutoEncoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.enc = encoder
        self.dec = decoder
    
    def forward(self, x):
        enc_out = self.enc(x)
        return self.dec(enc_out)

model_ae = AutoEncoder(encoder, decoder)

## Model Training

In [8]:
set_seed(13)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_ae.to(device)
loss_fn = nn.MSELoss()
optim = torch.optim.Adam(model_ae.parameters(), 0.0003)

num_epochs = 10

train_losses = []

for epoch in range(1, num_epochs+1):
    batch_losses = []
    for i, (x, _) in enumerate(circles_dl):
        model_ae.train()
        x = x.to(device)

        yhat = model_ae(x)
        loss = loss_fn(yhat, x)
        loss.backward()
        optim.step()
        optim.zero_grad()

        batch_losses.append(np.array([loss.data.item()]))
    train_losses.append(np.array(batch_losses).mean(axis=0))

    print(f"Epoch {epoch:03d} | Loss >> {train_losses[-1][0]:.4f}")

Epoch 001 | Loss >> 0.1388
Epoch 002 | Loss >> 0.0062
Epoch 003 | Loss >> 0.0049
Epoch 004 | Loss >> 0.0048
Epoch 005 | Loss >> 0.0048
Epoch 006 | Loss >> 0.0048
Epoch 007 | Loss >> 0.0048
Epoch 008 | Loss >> 0.0048
Epoch 009 | Loss >> 0.0047
Epoch 010 | Loss >> 0.0045
